# Synthetic fingerprint + iris fusion experiment

> **Important:** Fingerprint and iris source datasets contain different people. The deterministic pairing creates **synthetic virtual identities**, not genuine multimodal people.

This notebook exposes the complete fusion authentication artifact set: metrics, pairing metadata, score arrays, fusion deltas, and every generated plot category.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
RESULTS = ROOT / "results" / "experiments" / "fusion"
pairing = json.loads((RESULTS / "pairing.json").read_text())
metrics = json.loads((RESULTS / "metrics.json").read_text())
metrics_df = pd.DataFrame(metrics)
display(json.loads((RESULTS / "manifest.json").read_text()))

## 1. Fusion metrics and operating points

In [ ]:
print(pairing["warning"])
print("Selected virtual subjects:", pairing["selected_count"])
print("Trial count per model pair:", pairing["trial_count"])
display(metrics_df)

## 2. Fusion delta analysis

In [ ]:
delta = json.loads((RESULTS / "fusion_delta.json").read_text()) if (RESULTS / "fusion_delta.json").exists() else []
display(pd.DataFrame(delta))

## 3. Score artifacts

In [ ]:
rows=[]
for path in sorted((RESULTS / "scores").glob("*.npz")):
    with np.load(path) as data:
        rows.append({"artifact":path.name,"genuine_trials":int(data["genuine"].size),"impostor_trials":int(data["impostor"].size),"genuine_mean":float(data["genuine"].mean()),"impostor_mean":float(data["impostor"].mean())})
display(pd.DataFrame(rows))

## 4. All generated fusion plots

In [ ]:
headings = {
    "comparisons": "Comparisions Plots", 
    "det": "DET (Detection Error Tradeoff Plots)", 
    "distributions": "Score Distributions Plot", 
    "far_frr": "FAR_FRR_vs_Threshold Plots", 
    "pca": "PCA Plots", 
    "roc": "ROC Curve"
}

def show_plot_group(group, width=600):
    folder = RESULTS / "plots" / group
    print(f"\n### {headings.get(group, None)}")
    if not folder.exists():
        print("Not generated yet.")
        return
    for path in sorted(folder.glob("*.png")):
        print(path.name)
        display(Image(filename=str(path), width=width))

for group in ["comparisons", "det", "distributions", "far_frr", "pca", "roc"]:
    show_plot_group(group)
